# Homework 6: dlt + Logfire

Instrument the Module-1 FAQ agent (rewritten in Pydantic AI) with Pydantic Logfire, pull the trace data back out with `dlt`, and answer:

- **Q1** — how many spans does a single agent run produce for *"How do I run Ollama locally?"*
- **Q2** — how many tables does `dlt` create when loading the traces into DuckDB
- **Q3** — what's the input-token usage range for the Q1 agent run

Each answer is derived programmatically from live Logfire data, then matched to the nearest multiple-choice option.

In [1]:
import os
import sys
import time
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY, LOGFIRE_TOKEN, LOGFIRE_READ_TOKEN from .env

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
assert os.environ.get("LOGFIRE_TOKEN"), "Set LOGFIRE_TOKEN in .env (Q1)"
assert os.environ.get("LOGFIRE_READ_TOKEN"), "Set LOGFIRE_READ_TOKEN in .env (Q2)"

DUCKDB_PATH = os.path.join(os.getcwd(), "logfire_traces.duckdb")
QUESTION = "How do I run Ollama locally?"

OPTIONS_Q1 = [1, 5, 15, 30]
OPTIONS_Q2 = [1, 3, 24, 100]
OPTIONS_Q3 = [(100, 500), (1500, 5000), (10000, 20000), (50000, 100000)]


def nearest(value, options):
    return min(options, key=lambda o: abs(o - value))


def nearest_range(value, ranges):
    def dist(r):
        lo, hi = r
        if lo <= value <= hi:
            return 0
        return min(abs(value - lo), abs(value - hi))
    return min(ranges, key=dist)

## Question 1. Instrument the agent with Logfire

`logfire.configure()` + `logfire.instrument_pydantic_ai()` turn on tracing for every Pydantic AI agent in the process. We then run the agent several times with the exact question from the homework, flush after each run, and query Logfire's own Query API (`logfire.query_client.LogfireQueryClient`, auto-detects the EU/US region from the read token) to count spans per run — this avoids relying on the UI and gives a reproducible, programmatic count.

In [2]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()  # instruments every Pydantic AI agent from here on

from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data

documents = load_faq_data()
index = build_index(documents)
deps = SearchDeps(index=index)

Logfire project URL: https://logfire-eu.pydantic.dev/masses/llm-zoomcamp

In [3]:
# Run the agent several times with the exact Q1 question, recording the
# UTC time window of each run so we can look them up in Logfire afterwards.
# We use `.run()` (awaited) instead of `.run_sync()` since Jupyter already
# has its own running event loop, which `run_sync()` can't nest into.
N_RUNS = 5
run_windows = []

for i in range(N_RUNS):
    start = datetime.now(timezone.utc)
    result = await faq_agent.run(QUESTION, deps=deps)
    logfire.force_flush()
    end = datetime.now(timezone.utc)
    run_windows.append((start, end))
    print(f"run {i}: {(end - start).total_seconds():.1f}s")

12:46:15.345 faq_agent run
12:46:15.347   chat gpt-5.4-mini


12:46:16.816   running tool: search
12:46:16.821   chat gpt-5.4-mini


run 0: 3.6s
12:46:18.928 faq_agent run
12:46:18.930   chat gpt-5.4-mini


12:46:20.036   running tool: search
12:46:20.044   chat gpt-5.4-mini


run 1: 3.4s
12:46:22.329 faq_agent run
12:46:22.331   chat gpt-5.4-mini


...

run 2: 4.0s
12:46:26.350 faq_agent run
12:46:26.351   chat gpt-5.4-mini


12:46:27.382   running tool: search
12:46:27.392   chat gpt-5.4-mini


run 3: 3.1s
12:46:29.447 faq_agent run
12:46:29.451   chat gpt-5.4-mini


12:46:30.469   running tool: search
12:46:30.478   chat gpt-5.4-mini


run 4: 3.9s


In [4]:
from logfire.query_client import LogfireQueryClient

READ_TOKEN = os.environ["LOGFIRE_READ_TOKEN"]

overall_min = min(w[0] for w in run_windows) - timedelta(seconds=2)
overall_max = max(w[1] for w in run_windows) + timedelta(seconds=30)


def in_a_run_window(ts_str):
    ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
    return any(start - timedelta(seconds=1) <= ts <= end + timedelta(seconds=1) for start, end in run_windows)


# Logfire ingestion isn't instantaneous, so poll until all N_RUNS traces show
# up (or we give up after a couple of minutes).
q1_traces = []
with LogfireQueryClient(read_token=READ_TOKEN) as client:
    for attempt in range(12):
        rows = client.query_json_rows(
            """
            SELECT trace_id, COUNT(*) AS span_count, MIN(start_timestamp) AS trace_start
            FROM records
            GROUP BY trace_id
            ORDER BY trace_start
            """,
            min_timestamp=overall_min,
            max_timestamp=datetime.now(timezone.utc),
        )["rows"]
        q1_traces = [r for r in rows if in_a_run_window(r["trace_start"])]
        if len(q1_traces) >= N_RUNS:
            break
        time.sleep(10)

span_counts = [r["span_count"] for r in q1_traces]
q1_trace_ids = [r["trace_id"] for r in q1_traces]

print(f"found {len(q1_traces)}/{N_RUNS} traces")
print("span counts per run:", span_counts)

avg_spans = sum(span_counts) / len(span_counts)
answer_q1 = nearest(avg_spans, OPTIONS_Q1)
print(f"average spans per run: {avg_spans:.1f} -> nearest option: {answer_q1}")

found 5/5 traces
span counts per run: [4, 4, 6, 4, 4]
average spans per run: 4.4 -> nearest option: 5


## Question 2. Load traces into DuckDB with dlt

Rather than the declarative `rest_api` source (the Logfire Query API takes its SQL and time range as a POST body, which doesn't map cleanly onto that helper), we use a plain `@dlt.resource` generator backed by `logfire.query_client.LogfireQueryClient` — the same authenticated client used above — and let `dlt` do the normalization work. Each row's `attributes` column is deeply nested JSON (LLM messages, tool calls, token usage, instructions, etc.), so `dlt` will split every nested list into its own child table under the `agent_traces` dataset.

In [5]:
import dlt


@dlt.resource(name="records", write_disposition="replace")
def logfire_records():
    """Pull every span recorded in the last 7 days from Logfire's `records` table."""
    with LogfireQueryClient(read_token=READ_TOKEN) as client:
        result = client.query_json_rows(
            "SELECT * FROM records ORDER BY start_timestamp",
            min_timestamp=datetime.now(timezone.utc) - timedelta(days=7),
            max_timestamp=datetime.now(timezone.utc),
            limit=10_000,
        )
    yield from result["rows"]


pipeline = dlt.pipeline(
    pipeline_name="logfire_traces",
    destination=dlt.destinations.duckdb(DUCKDB_PATH),
    dataset_name="agent_traces",
)

load_info = pipeline.run(logfire_records())
print(load_info)

2026-07-21 12:46:55,097|[WARNING]|54116|8506940800|dlt|validate.py|verify_normalized_table:113|In schema `logfire_traces`: The following columns in table 'records' did not receive any data during this load and therefore could not have their types inferred:
  - attributes__model_request_parameters__output_object
  - attributes__model_request_parameters__prompted_output_template
  - attributes__model_request_parameters__thinking
  - deployment_environment
  - exception_message
  - exception_stacktrace
  - exception_type
  - http_method
  - http_response_status_code
  - http_route
  - log_body
  - otel_status_message
  - url_full
  - url_path
  - url_query

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'attributes__model_request_parameters__output_object': {'data_type': 'text'}})



2026-07-21 12:46:55,097|[WARNING]|54116|8506940800|dlt|validate.py|verify_normalized_table:113|In schema `logfire_traces`: The following columns in table 'records__attributes__model_request_parameters__function_tools' did not receive any data during this load and therefore could not have their types inferred:
  - capability_id
  - include_return_schema
  - metadata
  - outer_typed_dict_key
  - return_schema
  - timeout
  - tool_kind
  - unless_native
  - with_native

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'capability_id': {'data_type': 'text'}})



Pipeline logfire_traces load step completed in 0.36 seconds
1 load package(s) were loaded to destination duckdb and into dataset agent_traces
The duckdb destination used duckdb:////Users/abhirupghosh/Documents/Work/job_preparation/preparation/ds_courses/llm-zoomcamp/cohorts/2026/workshops/dlt/homework/logfire_traces.duckdb location to store data
Load package 1784630814.7321181 is LOADED and contains no failed jobs


In [6]:
import duckdb

con = duckdb.connect(DUCKDB_PATH, read_only=True)

n_tables = con.execute(
    "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'agent_traces'"
).fetchone()[0]

table_names = con.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'agent_traces' ORDER BY table_name"
).fetchall()

print(f"{n_tables} tables in agent_traces:")
for (name,) in table_names:
    print(" -", name)

answer_q2 = nearest(n_tables, OPTIONS_Q2)
print(f"\n{n_tables} tables -> nearest option: {answer_q2}")

24 tables in agent_traces:
 - _dlt_loads
 - _dlt_pipeline_state
 - _dlt_version
 - records
 - records__attributes__gen_ai_input_messages
 - records__attributes__gen_ai_input_messages__parts
 - records__attributes__gen_ai_input_messages__parts__result
 - records__attributes__gen_ai_output_messages
 - records__attributes__gen_ai_output_messages__parts
 - records__attributes__gen_ai_response_finish_reasons
 - records__attributes__gen_ai_system_instructions
 - records__attributes__gen_ai_tool_call_result
 - records__attributes__gen_ai_tool_definitions
 - records__attributes__gen_ai_tool_definitions__parameters__required
 - records__attributes__logfire_metrics__gen_ai_client_token_usage__details
 - records__attributes__logfire_metrics__operation_cost__details
 - records__attributes__logfire_scrubbed
 - records__attributes__logfire_scrubbed__path
 - records__attributes__model_request_parameters__function_tools
 - records__attributes__model_request_parameters__function_tools__parameters_json_

## Question 3. Query traces with an agent

`gen_ai.usage.input_tokens` lives as a flat column (`attributes__gen_ai_usage_input_tokens`) on the `records` table dlt just created, once per LLM-call span (`chat gpt-5.4-mini`). We sum it per `trace_id`, restricted to the Q1 trace IDs captured above, then check which of the four ranges the totals fall into.

In [7]:
placeholders = ", ".join(f"'{tid}'" for tid in q1_trace_ids)

token_sums = con.execute(
    f"""
    SELECT trace_id, SUM(attributes__gen_ai_usage_input_tokens) AS total_input_tokens
    FROM agent_traces.records
    WHERE trace_id IN ({placeholders})
      AND attributes__gen_ai_usage_input_tokens IS NOT NULL
    GROUP BY trace_id
    """
).fetchall()

totals = [t for _, t in token_sums]
print("input tokens per Q1 run:", totals)

avg_tokens = sum(totals) / len(totals)
answer_q3 = nearest_range(avg_tokens, OPTIONS_Q3)
print(f"average input tokens per run: {avg_tokens:.0f} -> nearest range: {answer_q3[0]} - {answer_q3[1]}")

input tokens per Q1 run: [1623, 1501, 4266, 1488, 1486]
average input tokens per run: 2073 -> nearest range: 1500 - 5000


In [8]:
print("Q1 (spans per run):        ", answer_q1)
print("Q2 (tables in agent_traces):", answer_q2)
print("Q3 (input tokens range):    ", f"{answer_q3[0]} - {answer_q3[1]}")

con.close()

Q1 (spans per run):         5
Q2 (tables in agent_traces): 24
Q3 (input tokens range):     1500 - 5000
